# TogoLM Fine-Tuning — Kaggle T4 (Unsloth)

Fine-tune Mistral 7B Instruct v0.3 on the TogoLM corpus using QLoRA via Unsloth.

**Setup checklist before running:**
- [ ] Accelerator set to GPU T4 x2 (Session options)
- [ ] Internet ON
- [ ] Dataset `togolm-sft` added (contains train + eval JSONL)
- [ ] Secret `HF_TOKEN` added with your HuggingFace token
- [ ] Mistral-7B-Instruct-v0.3 license accepted on HuggingFace

In [ ]:
# 1. Patch broken system packages + install dependencies + purge module cache
import os
import shutil
import sys

# Disable wandb before any import
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# ── Step 1: patch files that pip install won't touch ─────────────────────────
#
# bitsandbytes 0.43.x — imports triton.ops which no longer exists in newer triton
#
# wandb — proto files compiled against old protobuf (3.x vs required 4+).
#   Stub wandb/__init__.py so `import wandb` succeeds without loading any proto.
#   wandb is fully disabled (report_to="none") so stubs are never called.
WANDB_STUB = """\
class _Stub:
    def __getattr__(self, _): return _Stub()
    def __call__(self, *a, **kw): return _Stub()
    def __bool__(self): return False

run = None
config = _Stub()
Settings = _Stub
sdk = _Stub()
apis = _Stub()

def init(*a, **kw): return _Stub()
def log(*a, **kw): pass
def finish(*a, **kw): pass
def watch(*a, **kw): pass
def alert(*a, **kw): pass
def login(*a, **kw): return True
"""

for path, content in {
    "/usr/local/lib/python3.12/dist-packages/bitsandbytes/triton/int8_matmul_mixed_dequantize.py": "int8_matmul_mixed_dequantize = None\n",
    "/usr/local/lib/python3.12/dist-packages/bitsandbytes/triton/dequantize_rowwise.py": "dequantize_rowwise = None\n",
    "/usr/local/lib/python3.12/dist-packages/wandb/__init__.py": WANDB_STUB,
}.items():
    if os.path.exists(path):
        open(path, "w").write(content)
        print(f"Patched: {path}")

for d in ["/usr/local/lib/python3.12/dist-packages/wandb/__pycache__"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleared: {d}")

# ── Step 2: pip install (may overwrite unsloth — patch it AFTER) ──────────────
#   trl==0.8.6              : neftune_post_forward_hook removed in trl>=0.9.0
#   transformers==4.45.2    : stable version compatible with unsloth on Kaggle
#   huggingface_hub==0.23.4 : >=0.26 removed utils._token used by unsloth
!pip install -q unsloth "transformers==4.45.2" "trl==0.8.6" "huggingface_hub==0.23.4" datasets

# ── Step 3: patch unsloth AFTER pip install (so pip can't overwrite it) ───────
# unsloth 2024.9 references FP8BackendType from accelerate.utils, but recent
# accelerate versions renamed/removed it. The try/except import fails silently,
# leaving the name undefined → NameError at trainer.train().
# Prepend a guaranteed fallback definition.
_utils_path = "/usr/local/lib/python3.12/dist-packages/unsloth/models/_utils.py"
if os.path.exists(_utils_path):
    content = open(_utils_path).read()
    if "FP8BackendType" in content and "FP8BackendType = type(" not in content:
        fallback = (
            "try:\n"
            "    from accelerate.utils import FP8BackendType\n"
            "except (ImportError, AttributeError):\n"
            "    FP8BackendType = type('FP8BackendType', (), {})\n\n"
        )
        open(_utils_path, "w").write(fallback + content)
        print(f"Patched FP8BackendType: {_utils_path}")
    else:
        print("_utils.py: FP8BackendType already handled or not present")

for d in ["/usr/local/lib/python3.12/dist-packages/unsloth/models/__pycache__"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleared: {d}")

# ── Step 4: purge module cache ────────────────────────────────────────────────
purged = [
    k for k in list(sys.modules) if any(x in k for x in ("bitsandbytes", "unsloth", "wandb", "trl"))
]
for k in purged:
    del sys.modules[k]
print(f"Purged {len(purged)} cached modules")

print("Done — run all remaining cells in order.")

In [ ]:
# 2. Purge module cache, import unsloth, then inject missing symbols
import sys

# Re-purge after pip install to force Python to reload the patched files
purged = [k for k in list(sys.modules) if "bitsandbytes" in k or "unsloth" in k or "trl" in k]
for k in purged:
    del sys.modules[k]
print(f"Purged {len(purged)} cached modules")

# unsloth must be imported before transformers and trl.
# IMPORTANT: do NOT import accelerate before unsloth — accelerate pulls in
# bitsandbytes which registers PyTorch operators; unsloth then tries to register
# the same operators again → RuntimeError (duplicate operator registration).
import unsloth  # noqa: E402, F401

# Inject FP8BackendType into every loaded unsloth module's globals.
#
# The real accelerate.utils.FP8BackendType is an enum with .TE, .MSAMP, .AO.
# Recent accelerate versions renamed/removed it. Unsloth does a try/except
# import that fails silently → name undefined in module globals → NameError or
# AttributeError when a method does `if fp8_backend == FP8BackendType.TE:`.
#
# We ALWAYS overwrite (not just inject if missing) because a previous session
# may have left an attribute-less stub via the cell-1 file patch, which would
# survive in sys.modules and cause AttributeError on .TE/.AO access.
#
# fp8_backend is None in our TrainingArguments, so all comparisons return False.
_dummy_fp8 = type("FP8BackendType", (), {"TE": "te", "MSAMP": "msamp", "AO": "ao"})
_count = 0
for _mod_name, _mod in list(sys.modules.items()):
    if "unsloth" in _mod_name and hasattr(_mod, "__dict__"):
        _mod.__dict__["FP8BackendType"] = _dummy_fp8
        _count += 1
print(f"FP8BackendType patched in {_count} unsloth module(s)")

from huggingface_hub import login  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("HuggingFace login OK")

In [ ]:
# 3. Check GPU
import torch

for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {name} — {vram:.1f} GB")

In [ ]:
# 4. Locate dataset files
import glob

train_files = glob.glob("/kaggle/input/**/togolm_sft_train.jsonl", recursive=True)
eval_files = glob.glob("/kaggle/input/**/togolm_sft_eval.jsonl", recursive=True)

assert train_files, "togolm_sft_train.jsonl not found — did you add the dataset?"
assert eval_files, "togolm_sft_eval.jsonl not found"

TRAIN_FILE = train_files[0]
EVAL_FILE = eval_files[0]
print(f"Train : {TRAIN_FILE}")
print(f"Eval  : {EVAL_FILE}")

In [ ]:
# 5. Configuration
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "/kaggle/working/togolm-7b"
HF_REPO = "togolm/togolm-7b-instruct-v1"  # HuggingFace repo for the LoRA adapter
CORPUS_REPO = "togolm/togolm-corpus-v1"  # HuggingFace repo for the SFT dataset

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8  # effective batch size = 16
LR = 2e-4
MAX_SEQ_LEN = 2048

In [ ]:
# 6. Load model with Unsloth (4-bit QLoRA)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,  # 4-bit NF4 quantization to fit in T4 VRAM
    token=hf_token,
)

# Attach LoRA adapters to all attention and MLP projection layers
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth memory optimisation
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# 7. Load SFT dataset
import json

from datasets import Dataset, DatasetDict


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


# Alpaca format: instruction and response separated by section headers
ALPACA_TEMPLATE = """Below is an instruction about Togo. Write a response that answers it accurately.

### Instruction:
{instruction}

### Response:
{output}"""


# formatting_func receives a BATCH (dict of lists), not a list of dicts.
# e.g. example = {"instruction": ["q1", "q2", ...], "output": ["a1", "a2", ...]}
def formatting_fn(example):
    return [
        ALPACA_TEMPLATE.format(instruction=inst, output=out)
        for inst, out in zip(example["instruction"], example["output"])
    ]


dataset = DatasetDict(
    {
        "train": Dataset.from_list(read_jsonl(TRAIN_FILE)),
        "eval": Dataset.from_list(read_jsonl(EVAL_FILE)),
    }
)
print(f"Train: {len(dataset['train'])} examples | Eval: {len(dataset['eval'])} examples")

In [ ]:
# 8. Train
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=False,
    fp16=True,  # T4 does not support bf16
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",  # no wandb or tensorboard
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    formatting_func=formatting_fn,
    max_seq_length=MAX_SEQ_LEN,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
# 9. Save LoRA adapter locally
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Adapter saved to {OUTPUT_DIR}/final")

In [ ]:
# 10. Push adapter to HuggingFace Hub
if HF_REPO:
    model.push_to_hub(HF_REPO, token=hf_token)
    tokenizer.push_to_hub(HF_REPO, token=hf_token)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")
else:
    print("HF_REPO not set — download the adapter from /kaggle/working/togolm-7b/final")

In [ ]:
# 11. Upload SFT dataset to togolm/togolm-corpus-v1
from huggingface_hub import HfApi

api = HfApi(token=hf_token)

# Create the dataset repo if it doesn't exist yet
api.create_repo(
    repo_id=CORPUS_REPO,
    repo_type="dataset",
    exist_ok=True,
)
print(f"Dataset repo ready: huggingface.co/datasets/{CORPUS_REPO}")

for local_path, repo_path in [
    (TRAIN_FILE,  "sft/togolm_sft_train.jsonl"),
    (EVAL_FILE,   "sft/togolm_sft_eval.jsonl"),
]:
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=CORPUS_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded {repo_path} -> huggingface.co/datasets/{CORPUS_REPO}")

In [ ]:
# 12. Quick inference test on the trained model
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

question = "Quel est le taux d'imposition sur les sociétés au Togo ?"
prompt = f"### Instruction:\n{question}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))